In [ ]:
import serial
import numpy as np
import librosa
import tensorflow as tf
import joblib
import time

# -------------------------------
# LOAD MODEL
# -------------------------------
interpreter = tf.lite.Interpreter(
    model_path="cnn_model_retrain_optimized.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

label_encoder = joblib.load("label_encoder_copy.pkl")

# -------------------------------
# SERIAL SETUP
# -------------------------------
ser = serial.Serial("COM4", 115200, timeout=1)
time.sleep(2)

# -------------------------------
# IMPORTANT FIX: LARGE BUFFER
# -------------------------------
BUFFER_SIZE = 44100   # ~2 seconds audio

def predict_audio(samples):

    audio = np.array(samples, dtype=np.float32)

    # Normalize safely
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))

    # -------------------------------
    # ENERGY FILTER (NEW)
    # -------------------------------
    energy = np.mean(np.abs(audio))

    if energy < 0.02:
        print("Prediction: No significant sound")
        return 'x'

    # -------------------------------
    # MEL SPECTROGRAM
    # -------------------------------
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=22050,
        n_mels=128
    )

    mel_spec_db = librosa.power_to_db(mel_spec)

    mel_spec_db = librosa.util.fix_length(
        mel_spec_db,
        size=173,
        axis=1
    )

    mel_spec_db = np.expand_dims(mel_spec_db, axis=-1)
    mel_spec_db = np.expand_dims(mel_spec_db, axis=0)
    mel_spec_db = mel_spec_db.astype(np.float32)

    # -------------------------------
    # MODEL PREDICTION
    # -------------------------------
    interpreter.set_tensor(input_details[0]['index'], mel_spec_db)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])[0]

    predicted_class = np.argmax(output)
    confidence = output[predicted_class]

    label = label_encoder.classes_[predicted_class]

    # -------------------------------
    # CLASS-SPECIFIC THRESHOLDS (NEW)
    # -------------------------------
    CLASS_THRESHOLDS = {
        "car_horn": 0.75,
        "dog_bark": 0.60,
        "siren": 0.65
    }

    threshold = CLASS_THRESHOLDS[label]

    print("Probabilities:", output)

    if confidence > threshold:
        print("Prediction:", label)
        return predicted_class
    else:
        print("Prediction: Low confidence")
        return 'x'


# -------------------------------
# MAIN LOOP
# -------------------------------
print("Listening from ESP32...")

for cycle in range(10):

    samples = []

    while len(samples) < BUFFER_SIZE:
        try:
            line = ser.readline().decode().strip()
            if line:
                samples.append(int(line))
        except:
            pass

    if len(samples) == BUFFER_SIZE:

        samples_np = np.array(samples)

        print("Min:", np.min(samples_np), "Max:", np.max(samples_np))

        result = predict_audio(samples)

        # -------------------------------
        # SEND RESULT (UPDATED)
        # -------------------------------
        if result == 'x':
            ser.write(b'x')
        else:
            ser.write(str(result).encode())

print("Finished listening.")